<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set2/blob/main/DeepSeek_Set2__Selfrefine_ZeroShotBase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA NOT available. Using CPU.")

CUDA is available! Using GPU.
GPU device name: NVIDIA A100-SXM4-40GB


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.5Gi        77Gi       2.0Mi       4.2Gi        81Gi
Swap:             0B          0B          0B


In [ ]:
from google.colab import files
import zipfile
import torch

import os
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import shutil

# CLEANUP - Remove old folders before extraction
if os.path.exists('input_folder'):
    shutil.rmtree('input_folder')
if os.path.exists('output_folder'):
    shutil.rmtree('output_folder')


In [ ]:
import shutil
import os

# Delete EVERYTHING (folders AND old zips)
!rm -rf input_folder output_folder *.zip

print("✅ All cleaned up!")
print("\n📂 Current directory:")
!ls -la

✅ All cleaned up!

📂 Current directory:
total 16
drwxr-xr-x 1 root root 4096 May 12 13:35 .
drwxr-xr-x 1 root root 4096 May 16 16:40 ..
drwxr-xr-x 4 root root 4096 May 12 13:35 .config
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data


In [ ]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving C.zip to C.zip
Files extracted to 'input_folder/'


In [ ]:
# 3. Load model
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-instruct")
tokenizer.pad_token = tokenizer.eos_token  # Add this line


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def translate_batch(c_code_list):
    """
    Self-refine pipeline for DeepSeek:
    Step 1 — Initial translation (zero-shot)
    Step 2 — Refinement: show the model its output + rules, ask it to fix or resubmit
    Mirrors Llama self-refine structure but uses DeepSeek's chat template.
    """
    results = []

    system_prompt = """You are an expert code translator. Your ONLY task is to convert C code to C++ code.
Rules you MUST follow:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior."""

    for i, c_code in enumerate(c_code_list):
        print(f"Processing {i+1}/{len(c_code_list)}...")

        try:
            # ── STEP 1: Initial Translation ──────────────────────────────
            user_prompt = f"""Translate this C code to C++ code:

C Code:
{c_code}

C++ Code:"""

            messages_step1 = [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ]

            initial_translation = _run_single_deepseek(messages_step1)

            # ── STEP 2: Self-Refinement ───────────────────────────────────
            refinement_prompt = f"""Review your C++ translation against these rules:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior

Additionally verify:
- All logic and functionality from the original C code is preserved
- All variables, functions and structures are correctly translated
- The code would compile without errors in a standard C++ compiler

Your translation:
{initial_translation}

If your translation violates any rule, fix it and resubmit the corrected C++ code only.
If your translation is correct, resubmit it as is.

C++ Code:"""

            messages_step2 = [
                {"role": "system",    "content": system_prompt},
                {"role": "user",      "content": user_prompt},
                {"role": "assistant", "content": initial_translation},
                {"role": "user",      "content": refinement_prompt}
            ]

            final_translation = _run_single_deepseek(messages_step2)
            final_translation = final_translation.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()
            results.append(final_translation)

        except Exception as e:
            print(f"  ❌ Error on file {i+1}: {e}")
            results.append(f"// Translation failed: {e}")

    print(f"\n✅ Done. Total files processed: {len(results)}")
    return results


def _run_single_deepseek(messages, max_new_tokens=512):
    """
    Run a single chat-template inference pass for DeepSeek.
    Key difference from Llama: terminator token is eos only — no eot_id.
    """
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt",
        padding=False,
        add_special_tokens=False
    ).to(model.device)

    # ── DeepSeek terminator — simpler than Llama ──
    terminators = tokenizer.eos_token_id

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=False,
        repetition_penalty=1.3  # ← helps prevent DeepSeek looping
    )

    input_len = inputs['attention_mask'][0].sum().item()
    response = output[0][input_len:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()

In [ ]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")

                import gc
                gc.collect()
                torch.cuda.empty_cache()
                # Clear batch lists
                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")

Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/2048.c → output_folder/C/2048.cpp
Translated: input_folder/C/13545.c → output_folder/C/13545.cpp
Translated: input_folder/C/723.c → output_folder/C/723.cpp
Translated: input_folder/C/7320.c → output_folder/C/7320.cpp
Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/13653.c → output_folder/C/13653.cpp
Translated: input_folder/C/8947.c → output_folder/C/8947.cpp
Translated: input_folder/C/12814.c → output_folder/C/12814.cpp
Translated: input_folder/C/2074.c → output_folder/C/2074.cpp
Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/13513.c → output_folder/C/13513.cpp
Translated: input_folder/C/9367.c → output_folder/C/9367.cpp
Translated: input_folder/C/139.c → output_folder/C/139.cpp
Transla

In [ ]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [ ]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r output.zip output_folder
colab_files.download('output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
  adding: output_folder/ (stored 0%)
  adding: output_folder/C/ (stored 0%)
  adding: output_folder/C/13414.cpp (deflated 49%)
  adding: output_folder/C/1710.cpp (deflated 91%)
  adding: output_folder/C/13537.cpp (deflated 52%)
  adding: output_folder/C/264.cpp (deflated 48%)
  adding: output_folder/C/2142.cpp (deflated 51%)
  adding: output_folder/C/9094.cpp (deflated 48%)
  adding: output_folder/C/1973.cpp (deflated 56%)
  adding: output_folder/C/8760.cpp (deflated 51%)
  adding: output_folder/C/1623.cpp (deflated 48%)
  adding: output_folder/C/1687.cpp (deflated 43%)
  adding: output_folder/C/10557.cpp (deflated 50%)
  adding: output_folder/C/2538.cpp (deflated 45%)
  adding: output_folder/C/1705.cpp (deflated 34%)
  adding: output_folder/C/9703.cpp (deflated 42%)
  adding: output_folder/C/2.cpp (deflated 53%)
  adding: output_folder/C/2001.cpp (deflated 40%)
  adding: output_folder/C/9497.cpp (deflated 50%)
  adding: output_folder/C/7321.cpp (deflated 49%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.


In [ ]:
print(model.generation_config)


GenerationConfig {
  "bos_token_id": 32013,
  "eos_token_id": 32021
}

